In [39]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ! pip install outlines datasets

In [ ]:
import outlines
from outlines.samplers import greedy

In [ ]:
from PIL import Image

def load_and_resize_image(image_path, max_size=1024):
    """
    Load and resize an image while maintaining aspect ratio

    Args:
        image_path: Path to the image file
        max_size: Maximum dimension (width or height) of the output image

    Returns:
        PIL Image: Resized image
    """
    image = Image.open(image_path)

    # Get current dimensions
    width, height = image.size

    # Calculate scaling factor
    scale = min(max_size / width, max_size / height)

    # Only resize if image is larger than max_size
    if scale < 1:
        new_width = int(width * scale)
        new_height = int(height * scale)
        image = image.resize((new_width, new_height), Image.Resampling.LANCZOS)

    return image


In [ ]:
def get_messages(image, base_prompt):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    # The image is provided as a PIL Image object
                    "type": "image",
                    "image": image,
                },
                {
                    "type": "text",
                    "text": base_prompt
                },
            ],
        }
    ]
    return messages

In [ ]:
from transformers import AutoProcessor
from transformers import AutoModelForVision2Seq

vmodel_name = "HuggingFaceTB/SmolVLM-256M-Instruct"
model_class = AutoModelForVision2Seq

vmodel = outlines.models.transformers_vision(
    vmodel_name,
    model_class=model_class,
)

# Used for generating prompt
processor = AutoProcessor.from_pretrained(vmodel_name)

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel

class Transaction(BaseModel):
    Date: str
    Type: Literal['invoice', 'credit_note', 'Debit']
    Ref: str
    Amount: float
    Balance: float
        
class Invoice(BaseModel):
    closing_balance: Optional[float]
    transactions: list[Transaction]
        
extract_json = outlines.generate.json(
        vmodel,
        Invoice, 
        sampler=greedy()    
    )

In [82]:
base_prompt="""
You are an expert Invoice data extractor.
Create an invoice json based on the provided image.
Only return Closing Balance (total balance which can be null) and
a list of transctions with Date, Type, Ref, Amount, and Balance.
The invoice data should be in JSON format.
"""

In [ ]:
from pprint import pprint
for i in range(1,5):
    image = load_and_resize_image(f"./images/{i}.png",512)
    prompt = processor.apply_chat_template(
        get_messages(image,base_prompt=base_prompt), 
        tokenize=False, 
        add_generation_prompt=True
    )
    pprint(extract_json(prompt, [image]))
    display(image)
    print("-------")